In [1]:
# AI Sales Deal Guidance & Profitability Copilot

##This notebook contains the initial MVP for a sales-operations use case. It analyzes deal pricing, discount-policy compliance, expected profit, profit margin, required approvals, and recommended next actions.

In [2]:
import pandas as pd

print("Project setup completed successfully.")

Project setup completed successfully.


In [3]:
# ============================================================
# MODULE 1: IMPORTS AND SAMPLE SALES DEAL DATA
# ============================================================

# Pandas helps us organize and display deal information.
import pandas as pd


# Create sample sales opportunities.
# These are simulated records for demonstration purposes.
sales_data = [
    {
        "deal_id": "D001",
        "customer_name": "ABC Manufacturing",
        "product": "Teamcenter",
        "region": "Europe",
        "list_price": 100000,
        "discount": 20,
        "estimated_cost": 62000
    },
    {
        "deal_id": "D002",
        "customer_name": "Global Motors",
        "product": "NX",
        "region": "North America",
        "list_price": 180000,
        "discount": 12,
        "estimated_cost": 125000
    },
    {
        "deal_id": "D003",
        "customer_name": "Future Industries",
        "product": "Simcenter",
        "region": "Asia Pacific",
        "list_price": 140000,
        "discount": 27,
        "estimated_cost": 110000
    }
]


# Convert the sample data into a DataFrame.
sales_df = pd.DataFrame(sales_data)


# Display the deal table.
sales_df

,deal_id,customer_name,product,region,list_price,discount,estimated_cost
0,D001,ABC Manufacturing,Teamcenter,Europe,100000,20,62000
1,D002,Global Motors,NX,North America,180000,12,125000
2,D003,Future Industries,Simcenter,Asia Pacific,140000,27,110000


In [4]:
# ============================================================
# MODULE 2: BUSINESS POLICY CONFIGURATION
# ============================================================

policies = {
    # Maximum discount allowed by the global policy.
    "global_max_discount": 25,

    # Regional discount limits.
    "regional_discount_limits": {
        "North America": 25,
        "Europe": 18,
        "Asia Pacific": 20
    },

    # Minimum preferred profit margin.
    "minimum_profit_margin": 20,

    # Approval levels based on the proposed discount.
    "approval_levels": [
        {
            "minimum_discount": 0,
            "maximum_discount": 10,
            "approval": "No additional approval required"
        },
        {
            "minimum_discount": 10.01,
            "maximum_discount": 15,
            "approval": "Sales Manager approval required"
        },
        {
            "minimum_discount": 15.01,
            "maximum_discount": 25,
            "approval": "Sales Director approval required"
        },
        {
            "minimum_discount": 25.01,
            "maximum_discount": 100,
            "approval": "VP and Pricing Committee approval required"
        }
    ]
}


print("Business policies loaded successfully.")

Business policies loaded successfully.


In [5]:
# ============================================================
# MODULE 3: INPUT VALIDATION
# ============================================================

def validate_deal(deal: dict) -> dict:
    """
    Validate required fields and financial values.

    Parameters:
        deal: Dictionary containing sales-deal information.

    Returns:
        Dictionary containing validation status and errors.
    """

    required_fields = [
        "deal_id",
        "customer_name",
        "product",
        "region",
        "list_price",
        "discount",
        "estimated_cost"
    ]

    errors = []

    # Check whether required fields are missing or empty.
    for field in required_fields:
        value = deal.get(field)

        if value is None:
            errors.append(
                f"{field.replace('_', ' ').title()} is missing."
            )

        elif isinstance(value, str) and value.strip() == "":
            errors.append(
                f"{field.replace('_', ' ').title()} cannot be empty."
            )

    # Validate the financial values.
    list_price = float(deal.get("list_price", 0))
    discount = float(deal.get("discount", 0))
    estimated_cost = float(deal.get("estimated_cost", 0))

    if list_price <= 0:
        errors.append("List price must be greater than zero.")

    if discount < 0 or discount > 100:
        errors.append(
            "Discount must be between 0 and 100 percent."
        )

    if estimated_cost < 0:
        errors.append(
            "Estimated cost cannot be negative."
        )

    return {
        "is_valid": len(errors) == 0,
        "errors": errors
    }

In [6]:
validation_result = validate_deal(sales_data[0])

validation_result

{'is_valid': True, 'errors': []}

In [7]:
# ============================================================
# MODULE 4: PROFITABILITY CALCULATION
# ============================================================

def calculate_profitability(deal: dict) -> dict:
    """
    Calculate profitability after applying the proposed discount.

    Parameters:
        deal: Dictionary containing list price, discount,
              and estimated cost.

    Returns:
        Financial calculation results.
    """

    # Read financial values from the deal.
    list_price = float(deal.get("list_price", 0))
    discount = float(deal.get("discount", 0))
    estimated_cost = float(deal.get("estimated_cost", 0))

    # Calculate the monetary value of the discount.
    discount_amount = list_price * (discount / 100)

    # Calculate the final selling price.
    final_selling_price = list_price - discount_amount

    # Calculate the expected profit.
    expected_profit = final_selling_price - estimated_cost

    # Calculate the profit margin.
    if final_selling_price > 0:
        profit_margin = (
            expected_profit / final_selling_price
        ) * 100
    else:
        profit_margin = 0

    return {
        "list_price": round(list_price, 2),
        "discount_percentage": round(discount, 2),
        "discount_amount": round(discount_amount, 2),
        "final_selling_price": round(final_selling_price, 2),
        "estimated_cost": round(estimated_cost, 2),
        "expected_profit": round(expected_profit, 2),
        "profit_margin": round(profit_margin, 2)
    }

In [8]:
profitability_result = calculate_profitability(
    sales_data[0]
)

profitability_result

{'list_price': 100000.0,
 'discount_percentage': 20.0,
 'discount_amount': 20000.0,
 'final_selling_price': 80000.0,
 'estimated_cost': 62000.0,
 'expected_profit': 18000.0,
 'profit_margin': 22.5}

In [9]:
# ============================================================
# MODULE 5: APPROVAL-LEVEL CALCULATION
# ============================================================

def get_approval_level(discount: float) -> str:
    """
    Determine the approval level based on the discount.

    Parameters:
        discount: Proposed discount percentage.

    Returns:
        Approval guidance.
    """

    # Compare the discount against each approval range.
    for rule in policies["approval_levels"]:
        if (
            rule["minimum_discount"]
            <= discount
            <= rule["maximum_discount"]
        ):
            return rule["approval"]

    return "Approval rule not found"

In [10]:
approval_result = get_approval_level(
    sales_data[0]["discount"]
)

approval_result

'Sales Director approval required'

In [11]:
# ============================================================
# MODULE 6: PRICING-POLICY COMPLIANCE
# ============================================================

def check_policy_compliance(deal: dict) -> dict:
    """
    Compare the proposed discount with regional and global limits.

    Parameters:
        deal: Dictionary containing region and discount.

    Returns:
        Policy status, limits, and identified issues.
    """

    region = deal.get("region", "")
    discount = float(deal.get("discount", 0))

    # Read the global discount limit.
    global_limit = policies["global_max_discount"]

    # Read the regional limit.
    # If the region is not found, use the global limit.
    regional_limit = policies[
        "regional_discount_limits"
    ].get(region, global_limit)

    policy_issues = []

    # Check the regional discount limit.
    if discount > regional_limit:
        policy_issues.append(
            f"The proposed discount of {discount:.1f}% exceeds "
            f"the {region} regional limit of "
            f"{regional_limit}%."
        )

    # Check the global discount limit.
    if discount > global_limit:
        policy_issues.append(
            f"The proposed discount of {discount:.1f}% exceeds "
            f"the global limit of {global_limit}%."
        )

    # Determine the overall policy status.
    if policy_issues:
        policy_status = "Exception or Adjustment Required"
    else:
        policy_status = "Policy Compliant"

    return {
        "policy_status": policy_status,
        "regional_limit": regional_limit,
        "global_limit": global_limit,
        "policy_issues": policy_issues
    }

In [12]:
policy_result = check_policy_compliance(
    sales_data[0]
)

policy_result

{'policy_status': 'Exception or Adjustment Required',
 'regional_limit': 18,
 'global_limit': 25,
 'policy_issues': ['The proposed discount of 20.0% exceeds the Europe regional limit of 18%.']}

In [13]:
# ============================================================
# MODULE 7: PROFIT-MARGIN EVALUATION
# ============================================================

def evaluate_profit_margin(
    profitability_result: dict
) -> dict:
    """
    Compare the profit margin with the minimum preferred margin.

    Parameters:
        profitability_result: Output from the profitability
                              calculation.

    Returns:
        Margin status and explanation.
    """

    profit_margin = profitability_result["profit_margin"]
    minimum_margin = policies["minimum_profit_margin"]

    # Healthy margin.
    if profit_margin >= minimum_margin:
        margin_status = "Healthy Margin"

        margin_message = (
            f"The expected profit margin is "
            f"{profit_margin:.2f}%, which meets the "
            f"minimum preferred margin of {minimum_margin}%."
        )

    # Profitable, but below preferred margin.
    elif profit_margin > 0:
        margin_status = "Low Margin"

        margin_message = (
            f"The expected profit margin is "
            f"{profit_margin:.2f}%, which is below the "
            f"minimum preferred margin of {minimum_margin}%."
        )

    # Zero or negative profit.
    else:
        margin_status = "Unprofitable Deal"

        margin_message = (
            "The estimated cost is equal to or greater than "
            "the final selling price."
        )

    return {
        "margin_status": margin_status,
        "minimum_margin": minimum_margin,
        "margin_message": margin_message
    }

In [14]:
margin_result = evaluate_profit_margin(
    profitability_result
)

margin_result

{'margin_status': 'Healthy Margin',
 'minimum_margin': 20,
 'margin_message': 'The expected profit margin is 22.50%, which meets the minimum preferred margin of 20%.'}

In [15]:
# ============================================================
# MODULE 8: COMPLETE SALES DEAL ANALYZER
# ============================================================

def analyze_deal(deal: dict) -> dict:
    """
    Perform the complete sales-deal analysis.

    The analysis includes:
    - input validation,
    - discounted selling price,
    - expected profit,
    - profit margin,
    - pricing-policy compliance,
    - approval requirements,
    - recommended next action.
    """

    # Validate the deal first.
    validation = validate_deal(deal)

    # Stop the analysis if the input is invalid.
    if not validation["is_valid"]:
        return {
            "deal_id": deal.get("deal_id"),
            "customer_name": deal.get("customer_name"),
            "analysis_status": "Invalid Deal Data",
            "validation_errors": validation["errors"]
        }

    # Run the financial calculations.
    profitability = calculate_profitability(deal)

    # Check policy compliance.
    policy_check = check_policy_compliance(deal)

    # Identify the approval level.
    approval = get_approval_level(
        float(deal.get("discount", 0))
    )

    # Evaluate the resulting profit margin.
    margin_check = evaluate_profit_margin(
        profitability
    )

    recommendation_parts = []

    # Pricing-policy recommendation.
    if policy_check["policy_status"] == "Policy Compliant":
        recommendation_parts.append(
            "The proposed discount complies with the applicable "
            "regional and global pricing policies."
        )
    else:
        recommendation_parts.append(
            "The proposed discount exceeds an applicable policy "
            "limit. Reduce the discount or request a pricing exception."
        )

    # Profitability recommendation.
    if margin_check["margin_status"] == "Healthy Margin":
        recommendation_parts.append(
            "The expected profit margin remains financially healthy."
        )

    elif margin_check["margin_status"] == "Low Margin":
        recommendation_parts.append(
            "The deal remains profitable, but the margin is below "
            "the preferred minimum. Review the discount or delivery cost."
        )

    else:
        recommendation_parts.append(
            "The deal is currently unprofitable and should not "
            "proceed without pricing or cost changes."
        )

    # Approval recommendation.
    recommendation_parts.append(
        f"Approval guidance: {approval}."
    )

    return {
        "deal_id": deal.get("deal_id"),
        "customer_name": deal.get("customer_name"),
        "product": deal.get("product"),
        "region": deal.get("region"),
        "analysis_status": "Analysis Completed",

        "list_price": profitability["list_price"],
        "discount_percentage": profitability[
            "discount_percentage"
        ],
        "discount_amount": profitability["discount_amount"],
        "final_selling_price": profitability[
            "final_selling_price"
        ],
        "estimated_cost": profitability["estimated_cost"],
        "expected_profit": profitability["expected_profit"],
        "profit_margin": profitability["profit_margin"],

        "policy_status": policy_check["policy_status"],
        "regional_discount_limit": policy_check[
            "regional_limit"
        ],
        "global_discount_limit": policy_check[
            "global_limit"
        ],
        "policy_issues": policy_check["policy_issues"],

        "approval_required": approval,
        "margin_status": margin_check["margin_status"],
        "minimum_profit_margin": margin_check[
            "minimum_margin"
        ],
        "margin_message": margin_check["margin_message"],

        "recommendation": " ".join(
            recommendation_parts
        )
    }

In [16]:
result = analyze_deal(sales_data[0])

result

{'deal_id': 'D001',
 'customer_name': 'ABC Manufacturing',
 'product': 'Teamcenter',
 'region': 'Europe',
 'analysis_status': 'Analysis Completed',
 'list_price': 100000.0,
 'discount_percentage': 20.0,
 'discount_amount': 20000.0,
 'final_selling_price': 80000.0,
 'estimated_cost': 62000.0,
 'expected_profit': 18000.0,
 'profit_margin': 22.5,
 'policy_status': 'Exception or Adjustment Required',
 'regional_discount_limit': 18,
 'global_discount_limit': 25,
 'policy_issues': ['The proposed discount of 20.0% exceeds the Europe regional limit of 18%.'],
 'approval_required': 'Sales Director approval required',
 'margin_status': 'Healthy Margin',
 'minimum_profit_margin': 20,
 'margin_message': 'The expected profit margin is 22.50%, which meets the minimum preferred margin of 20%.',
 'recommendation': 'The proposed discount exceeds an applicable policy limit. Reduce the discount or request a pricing exception. The expected profit margin remains financially healthy. Approval guidance: Sale

In [17]:
# ============================================================
# MODULE 9: COPILOT-STYLE RESULT DISPLAY
# ============================================================

def display_deal_result(result: dict) -> None:
    """
    Display the deal analysis in a clean and readable format.
    """

    print("=" * 76)
    print("AI SALES DEAL GUIDANCE & PROFITABILITY COPILOT")
    print("=" * 76)

    # Handle invalid data separately.
    if result.get("analysis_status") == "Invalid Deal Data":
        print(f"Deal ID: {result.get('deal_id')}")
        print(f"Customer: {result.get('customer_name')}")
        print("\nValidation Errors:")

        for error in result.get("validation_errors", []):
            print(f"- {error}")

        print("=" * 76)
        return

    # Display general deal information.
    print(f"Deal ID: {result['deal_id']}")
    print(f"Customer: {result['customer_name']}")
    print(f"Product: {result['product']}")
    print(f"Region: {result['region']}")

    # Display financial analysis.
    print("\nFINANCIAL ANALYSIS")
    print("-" * 76)

    print(
        f"List Price: "
        f"${result['list_price']:,.2f}"
    )

    print(
        f"Proposed Discount: "
        f"{result['discount_percentage']:.2f}%"
    )

    print(
        f"Discount Amount: "
        f"${result['discount_amount']:,.2f}"
    )

    print(
        f"Final Selling Price: "
        f"${result['final_selling_price']:,.2f}"
    )

    print(
        f"Estimated Cost: "
        f"${result['estimated_cost']:,.2f}"
    )

    print(
        f"Expected Profit: "
        f"${result['expected_profit']:,.2f}"
    )

    print(
        f"Profit Margin: "
        f"{result['profit_margin']:.2f}%"
    )

    # Display pricing-policy analysis.
    print("\nPOLICY ANALYSIS")
    print("-" * 76)

    print(
        f"Policy Status: "
        f"{result['policy_status']}"
    )

    print(
        f"Regional Discount Limit: "
        f"{result['regional_discount_limit']}%"
    )

    print(
        f"Global Discount Limit: "
        f"{result['global_discount_limit']}%"
    )

    if result["policy_issues"]:
        print("\nPolicy Issues:")

        for issue in result["policy_issues"]:
            print(f"- {issue}")

    # Display approval and margin status.
    print("\nAPPROVAL AND PROFITABILITY")
    print("-" * 76)

    print(
        f"Required Approval: "
        f"{result['approval_required']}"
    )

    print(
        f"Margin Status: "
        f"{result['margin_status']}"
    )

    print(result["margin_message"])

    # Display final recommendation.
    print("\nCOPILOT RECOMMENDATION")
    print("-" * 76)

    print(result["recommendation"])

    print("=" * 76)

In [18]:
display_deal_result(result)

AI SALES DEAL GUIDANCE & PROFITABILITY COPILOT
Deal ID: D001
Customer: ABC Manufacturing
Product: Teamcenter
Region: Europe

FINANCIAL ANALYSIS
----------------------------------------------------------------------------
List Price: $100,000.00
Proposed Discount: 20.00%
Discount Amount: $20,000.00
Final Selling Price: $80,000.00
Estimated Cost: $62,000.00
Expected Profit: $18,000.00
Profit Margin: 22.50%

POLICY ANALYSIS
----------------------------------------------------------------------------
Policy Status: Exception or Adjustment Required
Regional Discount Limit: 18%
Global Discount Limit: 25%

Policy Issues:
- The proposed discount of 20.0% exceeds the Europe regional limit of 18%.

APPROVAL AND PROFITABILITY
----------------------------------------------------------------------------
Required Approval: Sales Director approval required
Margin Status: Healthy Margin
The expected profit margin is 22.50%, which meets the minimum preferred margin of 20%.

COPILOT RECOMMENDATION
-----

In [19]:
# ============================================================
# MODULE 10: PORTFOLIO-LEVEL DEAL SUMMARY
# ============================================================

all_results = []

# Analyze every sample deal.
for deal in sales_data:
    deal_result = analyze_deal(deal)
    all_results.append(deal_result)


# Convert the results into a DataFrame.
results_df = pd.DataFrame(all_results)


# Select the most useful fields for the summary.
portfolio_summary = results_df[
    [
        "deal_id",
        "customer_name",
        "product",
        "region",
        "discount_percentage",
        "final_selling_price",
        "expected_profit",
        "profit_margin",
        "policy_status",
        "margin_status",
        "approval_required"
    ]
]


portfolio_summary

,deal_id,customer_name,product,region,discount_percentage,final_selling_price,expected_profit,profit_margin,policy_status,margin_status,approval_required
0,D001,ABC Manufacturing,Teamcenter,Europe,20.0,80000.0,18000.0,22.50,Exception or Adjustment Required,Healthy Margin,Sales Director approval required
1,D002,Global Motors,NX,North America,12.0,158400.0,33400.0,21.09,Policy Compliant,Healthy Margin,Sales Manager approval required
2,D003,Future Industries,Simcenter,Asia Pacific,27.0,102200.0,-7800.0,-7.63,Exception or Adjustment Required,Unprofitable Deal,VP and Pricing Committee approval required
